# Motor Imagery Analysis — NT@B FA26

Exploratory analysis and visualization for left/right fist motor imagery classification.

## Setup

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import yaml

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / "src"))

from ntab_bci.data import load_dataset
from ntab_bci.model import MotorImageryClassifier

%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')

## Load Configuration and Data

In [ ]:
# Load config
with open("../config.yaml") as f:
    config = yaml.safe_load(f)

print(f"Subjects: {config['subjects']}")
print(f"Condition: {config['primary_condition']}")
print(f"Frequency band: {config['l_freq']}-{config['h_freq']} Hz")

In [ ]:
# Load dataset
X, y, subject_ids = load_dataset(
    subjects=config["subjects"],
    condition=config["primary_condition"],
    data_dir=config["data_dir"],
    tmin=config["tmin"],
    tmax=config["tmax"],
    l_freq=config["l_freq"],
    h_freq=config["h_freq"],
)

print(f"Data shape: {X.shape}")
print(f"Labels: {len(y)}")
print(f"Class balance: {(y==0).sum()} left, {(y==1).sum()} right")

## Visualize Sample Epochs

In [ ]:
# Plot average ERP for each class
left_mask = y == 0
right_mask = y == 1

# Pick a central channel (e.g., Cz at index 31 for 64-channel 10-10)
channel_idx = 31

left_avg = X[left_mask, channel_idx, :].mean(axis=0)
right_avg = X[right_mask, channel_idx, :].mean(axis=0)

times = np.linspace(config['tmin'], config['tmax'], X.shape[2])

plt.figure(figsize=(12, 4))
plt.plot(times, left_avg, label='Left fist', alpha=0.7)
plt.plot(times, right_avg, label='Right fist', alpha=0.7)
plt.xlabel('Time (s)')
plt.ylabel('Amplitude (µV)')
plt.title(f'Average ERP at channel {channel_idx} (approx. Cz)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Train Model and Extract CSP Patterns

In [ ]:
# Train model
model = MotorImageryClassifier(
    n_components=config["n_csp_components"],
    reg=config["csp_reg"],
    random_state=config["random_seed"],
)
model.fit(X, y)

# Extract CSP features
features = model.transform_features(X)
print(f"CSP features shape: {features.shape}")

In [ ]:
# Visualize CSP feature distributions
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.ravel()

for i in range(min(6, features.shape[1])):
    axes[i].hist(features[left_mask, i], bins=20, alpha=0.6, label='Left', density=True)
    axes[i].hist(features[right_mask, i], bins=20, alpha=0.6, label='Right', density=True)
    axes[i].set_xlabel(f'CSP {i}')
    axes[i].set_ylabel('Density')
    axes[i].legend()
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.suptitle('CSP Feature Distributions', y=1.02, fontsize=14)
plt.show()

## Per-Subject Analysis

In [ ]:
# Class balance per subject
unique_subjects = np.unique(subject_ids)
for subj in unique_subjects:
    subj_mask = subject_ids == subj
    left_count = (y[subj_mask] == 0).sum()
    right_count = (y[subj_mask] == 1).sum()
    print(f"S{subj:03d}: {left_count} left, {right_count} right")

## Weakest Point Exploration

What do I trust least in this work?

**Small sample size:** ~40 epochs per subject across 3 runs. High variance, easy to overfit.

Let's check if any subjects have suspiciously different distributions:

In [ ]:
# Compute per-subject mean CSP feature values
for subj in unique_subjects:
    subj_mask = subject_ids == subj
    subj_features = features[subj_mask]
    print(f"S{subj:03d} CSP feature means: {subj_features.mean(axis=0)}")

## Next Steps

1. **More subjects:** Expand to 20+ for better cross-subject generalization
2. **Artifact rejection:** Remove eye blinks and muscle artifacts
3. **Time-frequency analysis:** Visualize mu/beta ERD/ERS patterns
4. **Spatial patterns:** Plot CSP topographies on scalp maps
5. **Hyperparameter tuning:** Grid search for optimal CSP components and frequency bands